# Batoid Pupil Comparison: Hexapod Defocus Configurations

**Author:** Aaron Roodman
**Date Created:** 2026-06-15
**Last Modified:** 2026-06-15
**Status:** In Progress
**Keywords:** batoid, pupil, donut, hexapod, M2, defocus, FAM, ray trace, spot diagram, AOS

## Description

Use **Batoid** + the LSST optical model to study how the giant-donut **pupil** depends on
*how* the defocus is produced, at the field position of our real giant donut on
**R30_S21** (pixel (1167, 2915)).

Method: trace a **dense grid of rays** (a spot diagram, many spots) from a point source
at that field angle through the defocused system to the detector; the 2-D **histogram of
ray positions is the geometric pupil/donut image**.

Comparisons:
1. **8 mm defocus, extra- and intra-focal:** Camera hexapod alone (±8 mm) vs Camera +
   M2 hexapods ±4 mm each (M2 is powered → contributes differently).
2. **FAM mode:** Camera hexapod ±1.5 mm — the intra- and extra-focal pupils used for
   curvature wavefront sensing.

Hexapod pistons are rigid z-shifts of the `LSSTCamera` group and `M2`
(`withGloballyShiftedOptic`; no `batoid_rubin` data download). Models: design
`LSST_r.yaml`, or as-built `Rubin_v3.12_r.yaml` / `Rubin_v3.14_r.yaml`.

**Output:** comparison figures; no files written by default.

**Based on:** the giant-donut study (`wfs_giant_donut_fit.ipynb`); R30_S21 seq 340
(intra_8mm) / 349 (extra_8mm).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-15 | Aaron Roodman | Initial version: batoid spot-diagram pupil, camera-8mm vs camera-4mm+M2-4mm |
| 2026-06-15 | Aaron Roodman | Field angle from the R30_S21 donut pixel; add intra-focal + FAM (±1.5 mm) comparisons |
| 2026-06-15 | Aaron Roodman | FAM: invert the extra donut (x→−x, y→−y) so intra/extra overlay (residual std 2.08→1.18) |
| 2026-06-15 | Aaron Roodman | §7: extract danish's fit pupil (DonutFactory+SingleDonutModel from the ts_wep Instrument, z_fit=0, full off-axis z_ref) and overlay on the batoid donut (giant camera-only 8 mm + FAM 1.5 mm) with difference + radial profiles |
| 2026-06-15 | Aaron Roodman | §8 rim radius vs field angle (danish vs batoid, inner & outer); §9 per-surface vignetting attribution via batoid traceFull (off-axis loss dominated by M1/M2/M3 edges + filter) |
| 2026-06-15 | Aaron Roodman | §10 danish mask vs batoid vignetting boundary in the ts_wep stop-surface pupil frame: ~98.6% agree at R30; inner edge exact, outer edge non-circular and danish (circle) over-extends mean +16 / max +90 mm (filter, then M1) — the pupil-plane origin of the donut-fit rim residual |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Field Angle of the R30_S21 Donut](#field)
5. [8 mm Defocus: Camera vs Camera+M2 (intra & extra)](#eightmm)
6. [FAM Mode: Intra vs Extra (±1.5 mm)](#fam)
7. [Danish Pupil vs Batoid Spot Diagram](#danish)
8. [Rim Mismatch vs Field Angle](#rimfield)
9. [What Vignettes the Off-Axis Beam?](#vignframe)
10. [Danish Mask vs Batoid Vignetting Boundary](#maskedge)


<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# LSST optical model: "LSST_r.yaml" (design ~v3.3),
#   "Rubin_v3.12_r.yaml" (as-built, default), "Rubin_v3.14_r.yaml" (as-built).
model_yaml = "Rubin_v3.12_r.yaml"
wavelength = 620e-9              # metres (r-band ~620 nm)

# Field position: the real giant donut on R30_S21. theta is computed from this pixel
# (set theta_x_deg / theta_y_deg directly to override).
detector_name = "R30_S21"
donut_pixel = (1167, 2915)       # (x, y) px
theta_x_deg = None               # None -> from the pixel above
theta_y_deg = None

# Defocus magnitudes (mm)
defocus_mm = 8.0                 # camera-only defocus
m2_split_mm = 4.0                # split: camera + M2 each this much (-> 8 mm total move)
fam_offset_mm = 1.5              # FAM camera-hexapod offset (curvature sensing)

# Ray grid / histogram
n_pupil = 700                    # rays across the pupil grid (n_pupil^2 spots)
hist_npix = 256                  # output donut image size (pixels)
half_mm_8mm = 4.0                # half-width for the 8 mm donut panels (mm)
half_mm_fam = 1.0                # half-width for the FAM donut panels (mm)

cmap = "viridis"


<a id='setup'></a>
## Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import batoid

import lsst.geom
from lsst.afw.cameraGeom import FIELD_ANGLE, PIXELS
from lsst.obs.lsst import LsstCam

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
camera = LsstCam.getCamera()


<a id='functions'></a>
## Helper Functions

In [ ]:
def field_angle_from_pixel(detector_name, x, y):
    """Field angle (deg) for a detector pixel, from LSST camera geometry.

    Note: gives the correct field *radius*; the absolute orientation vs batoid's
    field-angle frame is approximate (fine for comparing configs at one field point).
    """
    fa = camera[detector_name].getTransform(PIXELS, FIELD_ANGLE).applyForward(
        lsst.geom.Point2D(x, y))
    return float(np.degrees(fa.getX())), float(np.degrees(fa.getY()))


def build_telescope(fiducial, camera_dz_mm, m2_dz_mm):
    """Apply camera- and M2-hexapod z pistons (signed mm) as rigid global z-shifts.
    +z = extra-focal sense, -z = intra-focal (per the chosen convention)."""
    tel = fiducial
    if camera_dz_mm:
        tel = tel.withGloballyShiftedOptic("LSSTCamera", [0, 0, camera_dz_mm * 1e-3])
    if m2_dz_mm:
        tel = tel.withGloballyShiftedOptic("M2", [0, 0, m2_dz_mm * 1e-3])
    return tel


def trace_donut(tel, theta_x_deg, theta_y_deg, wavelength, n_pupil, npix, half_mm):
    """Trace a dense pupil ray grid; histogram (un-vignetted) positions -> donut image,
    centred on the donut centroid. Returns dict(image, span_mm, n_rays)."""
    rays = batoid.RayVector.asGrid(
        optic=tel, wavelength=wavelength,
        theta_x=np.deg2rad(theta_x_deg), theta_y=np.deg2rad(theta_y_deg), nx=n_pupil)
    tel.trace(rays)
    ok = ~rays.vignetted & ~rays.failed
    x = rays.x[ok] * 1e3; y = rays.y[ok] * 1e3
    cx, cy = x.mean(), y.mean()
    H, _, _ = np.histogram2d(x, y, bins=npix,
                             range=[[cx - half_mm, cx + half_mm],
                                    [cy - half_mm, cy + half_mm]])
    return {"image": H.T, "span_mm": float(x.max() - x.min()), "n_rays": int(ok.sum())}


def trace_configs(fiducial, configs, theta, n_pupil, npix, half_mm):
    """configs: list of (label, camera_dz_mm, m2_dz_mm). Returns list of (label, donut)."""
    out = []
    for label, cdz, mdz in configs:
        tel = build_telescope(fiducial, cdz, mdz)
        out.append((label, trace_donut(tel, theta[0], theta[1], wavelength,
                                        n_pupil, npix, half_mm)))
    return out


def azimuthal_profile(image, half_mm, nbin=120):
    """Azimuthally averaged radial profile of a centred donut image -> (r_mm, value)."""
    n = image.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    c = (n - 1) / 2.0
    r = np.hypot(xx - c, yy - c) * (2 * half_mm / n)
    edges = np.linspace(0, half_mm, nbin + 1)
    idx = np.digitize(r.ravel(), edges) - 1
    v = image.ravel()
    rc, prof = [], []
    for k in range(nbin):
        sel = idx == k
        if sel.any():
            rc.append(0.5 * (edges[k] + edges[k + 1])); prof.append(v[sel].mean())
    return np.array(rc), np.array(prof)


def show_pair_with_diff(ax_a, ax_b, ax_d, da, db, half_mm, labels):
    """Plot donut A, donut B, and A-B on three axes."""
    vmax = max(da["image"].max(), db["image"].max())
    ext = [-half_mm, half_mm, -half_mm, half_mm]
    for ax, d, lab in ((ax_a, da, labels[0]), (ax_b, db, labels[1])):
        ax.imshow(d["image"], origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
        ax.set_title(f"{lab}\nspan {d['span_mm']:.2f} mm", fontsize=9)
        ax.set_xlabel("mm"); ax.set_ylabel("mm")
    diff = da["image"] - db["image"]
    v = np.nanpercentile(np.abs(diff), 99.5)
    im = ax_d.imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    ax_d.set_title(f"({labels[0]}) − ({labels[1]})", fontsize=9)
    ax_d.set_xlabel("mm"); ax_d.set_ylabel("mm")
    return im


<a id='field'></a>
## Field Angle of the R30_S21 Donut

In [ ]:
if theta_x_deg is None or theta_y_deg is None:
    tx, ty = field_angle_from_pixel(detector_name, *donut_pixel)
else:
    tx, ty = theta_x_deg, theta_y_deg
theta = (tx, ty)
print(f"{detector_name} pixel {donut_pixel}: field angle = ({tx:.4f}, {ty:.4f}) deg, "
      f"radius = {np.hypot(tx, ty):.4f} deg")
print(f"model: {model_yaml}")


<a id='eightmm'></a>
## 8 mm Defocus: Camera vs Camera+M2 (intra & extra)

In [ ]:
fiducial = batoid.Optic.fromYaml(model_yaml)

configs_8mm = [
    ("extra: cam +8",            +defocus_mm, 0.0),
    ("extra: cam +4 + M2 +4",    +m2_split_mm, +m2_split_mm),
    ("intra: cam -8",            -defocus_mm, 0.0),
    ("intra: cam -4 + M2 -4",    -m2_split_mm, -m2_split_mm),
]
d8 = dict(trace_configs(fiducial, configs_8mm, theta, n_pupil, hist_npix, half_mm_8mm))
for lab, d in d8.items():
    print(f"  {lab:>24}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Rows: extra, intra.  Cols: camera-only, camera+M2, difference.
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
show_pair_with_diff(axes[0, 0], axes[0, 1], axes[0, 2],
                    d8["extra: cam +8"], d8["extra: cam +4 + M2 +4"], half_mm_8mm,
                    ("extra: cam +8", "extra: cam +4 + M2 +4"))
show_pair_with_diff(axes[1, 0], axes[1, 1], axes[1, 2],
                    d8["intra: cam -8"], d8["intra: cam -4 + M2 -4"], half_mm_8mm,
                    ("intra: cam -8", "intra: cam -4 + M2 -4"))
fig.suptitle(f"8 mm defocus pupil — {model_yaml}, field ({tx:.2f}, {ty:.2f})°  "
             f"[camera-only vs camera+M2]", fontsize=12)
plt.show()


In [ ]:
# Azimuthally averaged radial profiles for the 8 mm configs.
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, side in zip(axs, ("extra", "intra")):
    for lab, d in d8.items():
        if lab.startswith(side):
            r, p = azimuthal_profile(d["image"], half_mm_8mm)
            ax.plot(r, p, label=lab)
    ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
    ax.set_title(f"{side}-focal radial profile"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()


<a id='fam'></a>
## FAM Mode: Intra vs Extra (±1.5 mm)

In [ ]:
configs_fam = [
    ("FAM intra: cam -1.5", -fam_offset_mm, 0.0),
    ("FAM extra: cam +1.5", +fam_offset_mm, 0.0),
]
dfam = dict(trace_configs(fiducial, configs_fam, theta, n_pupil, hist_npix, half_mm_fam))
for lab, d in dfam.items():
    print(f"  {lab:>22}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Intra and extra are related by a parity flip, so invert the extra donut
# (x -> -x, y -> -y, i.e. a 180° point reflection of the centred image) to overlay it
# on the intra donut. The difference then shows the genuine intra/extra mismatch.
extra = dfam["FAM extra: cam +1.5"]
extra_inv = {"image": extra["image"][::-1, ::-1], "span_mm": extra["span_mm"]}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
show_pair_with_diff(axes[0], axes[1], axes[2],
                    dfam["FAM intra: cam -1.5"], extra_inv, half_mm_fam,
                    ("FAM intra: cam -1.5", "FAM extra: cam +1.5 (inverted x,y)"))
fig.suptitle(f"FAM-mode pupil (camera ±{fam_offset_mm} mm) — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°  [extra inverted to match intra]", fontsize=11)
plt.show()

# Radial profiles are azimuthal averages (flip-invariant), so compare directly.
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for lab, d in dfam.items():
    r, p = azimuthal_profile(d["image"], half_mm_fam)
    ax.plot(r, p, label=lab)
ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
ax.set_title("FAM intra vs extra radial profile"); ax.grid(alpha=0.3); ax.legend()
plt.show()


<a id='danish'></a>
## Danish Pupil vs Batoid Spot Diagram

Extract the **pupil danish actually uses in its fit** and overlay it on the batoid
spot-diagram donut. danish has no method to return the pupil, so we build its
`DonutFactory` + `SingleDonutModel` from the **ts_wep `Instrument`** (pupil obscuration
`R_outer`/`R_inner`, `mask_params` for M1/M2/M3/L1/Filter/spider, focal length), set
`z_ref` to the instrument **off-axis Zernikes** (which carry the defocus + design
off-axis aberration), use **no fitted Zernikes** (`z_fit=()`) and a tiny `fwhm` (0.3″),
and draw — giving danish's geometric donut/pupil.

Per **Josh Meyers**, danish/ts_wep assume the giant-donut defocus is **camera-only**, so
the danish giant pupil is compared to the batoid **camera-only +8 mm** donut (not the
camera-4 + M2-4 that physically produced the data — that mismatch is the A−B difference
in §5). danish and batoid share the field-angle convention, so the donuts register
directly (identity, no flip). Both images are unit-sum normalised; the difference shows
where danish's parametric pupil departs from batoid's full ray trace (vignetting/baffles).


In [ ]:
import danish
from lsst.ts.wep.utils import getTaskInstrument, DefocalType

compare_npix = 255   # danish requires odd npix; common grid for danish & batoid
danish_fwhm = 0.3    # arcsec; tiny blur -> sharp danish pupil edges


def danish_pupil_donut(detector_name, defocal_mm, defocal_type, theta_deg, npix,
                       half_mm, fwhm=0.3):
    """Geometric donut/pupil danish uses in its fit, at the given defocus, on a ±half_mm
    grid (unit-sum normalised). Builds danish's DonutFactory + SingleDonutModel from the
    ts_wep Instrument; z_ref = off-axis coeffs (carry defocus + design off-axis), and no
    fitted Zernikes (z_fit=())."""
    inst = getTaskInstrument("LSSTCam", detector_name)
    inst.defocalOffset = defocal_mm * 1e-3
    txd, tyd = theta_deg
    z_ref = inst.getOffAxisCoeff(txd, tyd, defocal_type, "r",
                                 nollIndicesModel=np.arange(79),
                                 nollIndicesIntr=np.arange(4, 23))
    pscale = 2 * half_mm * 1e-3 / npix          # m/pixel so the image spans ±half_mm
    factory = danish.DonutFactory(
        R_outer=inst.radius, R_inner=inst.radius * inst.obscuration,
        mask_params=inst.maskParams, focal_length=inst.focalLength, pixel_scale=pscale)
    model = danish.SingleDonutModel(factory, z_ref=z_ref, z_terms=(),
                                    thx=np.deg2rad(txd), thy=np.deg2rad(tyd), npix=npix)
    img = model.model(flux=1.0, dx=0.0, dy=0.0, fwhm=fwhm, z_fit=())
    return img / img.sum()


def batoid_norm(cam_dz_mm, m2_dz_mm, half_mm, npix):
    """Batoid spot-diagram donut on a ±half_mm grid, unit-sum normalised."""
    tel = build_telescope(fiducial, cam_dz_mm, m2_dz_mm)
    b = trace_donut(tel, tx, ty, wavelength, n_pupil, npix, half_mm)
    return b["image"] / b["image"].sum()


# danish assumes CAMERA-ONLY defocus -> compare to batoid camera-only.
comparisons = [
    ("giant: camera-only +8 mm", defocus_mm,   DefocalType.Extra, half_mm_8mm),
    ("FAM: camera +1.5 mm",      fam_offset_mm, DefocalType.Extra, half_mm_fam),
]
results = []
for label, cdz, dtype, half in comparisons:
    B = batoid_norm(cdz, 0.0, half, compare_npix)
    D = danish_pupil_donut(detector_name, cdz, dtype, theta, compare_npix, half, danish_fwhm)
    results.append((label, half, B, D))

fig, axes = plt.subplots(len(results), 3, figsize=(13, 4.6 * len(results)),
                         constrained_layout=True)
axes = np.atleast_2d(axes)
for row, (label, half, B, D) in enumerate(results):
    ext = [-half, half, -half, half]; vmax = max(B.max(), D.max())
    axes[row, 0].imshow(B, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 0].set_title(f"batoid — {label}", fontsize=9)
    axes[row, 1].imshow(D, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 1].set_title("danish pupil (z_fit=0)", fontsize=9)
    diff = B - D; v = np.nanpercentile(np.abs(diff), 99.5)
    axes[row, 2].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    axes[row, 2].set_title("batoid − danish", fontsize=9)
    for a in axes[row]:
        a.set_xlabel("mm"); a.set_ylabel("mm")
fig.suptitle(f"Danish pupil vs batoid spot diagram — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°", fontsize=12)
plt.show()

# Azimuthally averaged radial profiles (orientation-independent).
fig, axs = plt.subplots(1, len(results), figsize=(7 * len(results), 4.4),
                        constrained_layout=True)
axs = np.atleast_1d(axs)
for ax, (label, half, B, D) in zip(axs, results):
    rB, pB = azimuthal_profile(B, half); rD, pD = azimuthal_profile(D, half)
    ax.plot(rB, pB, label="batoid"); ax.plot(rD, pD, label="danish")
    ax.set_title(label); ax.set_xlabel("radius [mm]"); ax.set_ylabel("norm. illumination")
    ax.grid(alpha=0.3); ax.legend()
plt.show()


<a id='rimfield'></a>
## Rim Mismatch vs Field Angle

The danish and batoid pupils agree to within a pixel on-axis but **diverge off-axis** — and
that field-dependent rim mismatch is the prime suspect for the inner/outer-rim residuals
seen in the WFS/FAM donut fits (which all sit at field positions, the corners most of all).

We sweep the field radius from 0 to the R30_S21 edge (~1.76°, along its azimuth) and
measure the **inner and outer rim radii** (half-max of the azimuthal profile, sub-bin
interpolated) of the batoid spot-diagram donut and danish's fit pupil, for both the giant
(camera +8 mm) and FAM (camera +1.5 mm) camera-only defocus. The bottom row is the
**danish − batoid** rim difference in µm: ~0 to ~0.6°, then growing to a few tens of µm
at the field edge.

In [ ]:
# ------------------------------------------------------------------
# Rim radius vs field angle: where does danish depart from batoid?
# (depends on the §7 helpers danish_pupil_donut / compare_npix / danish_fwhm)
# ------------------------------------------------------------------
def _edge_crossings(r, p):
    """Inner & outer half-max radii of a centred donut radial profile, linearly
    interpolated between bins for sub-bin precision."""
    hm = 0.5 * p.max()
    above = np.where(p > hm)[0]
    if len(above) == 0:
        return np.nan, np.nan
    i0, i1 = above.min(), above.max()

    def cross(ia, ib):                       # interpolate to p == hm between bins ia, ib
        if ia < 0 or ib >= len(p) or p[ib] == p[ia]:
            return r[np.clip(ia if ia >= 0 else ib, 0, len(r) - 1)]
        return r[ia] + (hm - p[ia]) * (r[ib] - r[ia]) / (p[ib] - p[ia])

    return cross(i0 - 1, i0), cross(i1 + 1, i1)   # rising (inner), falling (outer)


def rim_vs_field(cam_dz_mm, dtype, half_mm, fields_deg, nbin=220):
    """Inner & outer rim radii of the batoid and danish camera-only donuts at each field
    radius (along the R30_S21 azimuth). Returns rows of (field_deg, bi, bo, di, do) mm."""
    az = np.arctan2(ty, tx)                  # R30_S21 azimuth
    rows = []
    for R in fields_deg:
        txd, tyd = R * np.cos(az), R * np.sin(az)
        tel = build_telescope(fiducial, cam_dz_mm, 0.0)
        Bimg = trace_donut(tel, txd, tyd, wavelength, n_pupil, hist_npix, half_mm)["image"]
        Dimg = danish_pupil_donut(detector_name, cam_dz_mm, dtype, (txd, tyd),
                                  compare_npix, half_mm, danish_fwhm)
        bi, bo = _edge_crossings(*azimuthal_profile(Bimg, half_mm, nbin=nbin))
        di, do = _edge_crossings(*azimuthal_profile(Dimg, half_mm, nbin=nbin))
        rows.append((R, bi, bo, di, do))
    return np.array(rows)


sweep_fields = np.array([0.0, 0.3, 0.6, 0.9, 1.2, 1.5, 1.76])
sweep = {
    "giant (camera +8 mm)": rim_vs_field(defocus_mm,   DefocalType.Extra, half_mm_8mm, sweep_fields),
    "FAM (camera +1.5 mm)": rim_vs_field(fam_offset_mm, DefocalType.Extra, half_mm_fam, sweep_fields),
}

fig, axes = plt.subplots(2, len(sweep), figsize=(7 * len(sweep), 8.5),
                         constrained_layout=True)
for col, (label, S) in enumerate(sweep.items()):
    R, bi, bo, di, do = S.T
    ax = axes[0, col]
    ax.plot(R, bi, "o-",  color="C0", label="batoid inner")
    ax.plot(R, bo, "o-",  color="C1", label="batoid outer")
    ax.plot(R, di, "s--", color="C0", mfc="none", label="danish inner")
    ax.plot(R, do, "s--", color="C1", mfc="none", label="danish outer")
    ax.set_title(label); ax.set_xlabel("field radius [deg]"); ax.set_ylabel("rim radius [mm]")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    axd = axes[1, col]
    axd.axhline(0, color="k", lw=0.6)
    axd.plot(R, (di - bi) * 1e3, "s-", color="C0", label="inner Δ")
    axd.plot(R, (do - bo) * 1e3, "s-", color="C1", label="outer Δ")
    axd.set_xlabel("field radius [deg]"); axd.set_ylabel("danish − batoid [µm]")
    axd.grid(alpha=0.3); axd.legend(fontsize=8)
fig.suptitle("Pupil rim radius vs field angle (camera-only defocus, R30_S21 azimuth)",
             fontsize=12)
plt.show()

for label, S in sweep.items():
    R, bi, bo, di, do = S.T
    print(f"{label}: at {R[-1]:.2f}°  inner Δ={1e3*(di[-1]-bi[-1]):+.1f} µm, "
          f"outer Δ={1e3*(do[-1]-bo[-1]):+.1f} µm   (on-axis Δ≈0)")


<a id='vignframe'></a>
## What Vignettes the Off-Axis Beam?

To find the *physical source* of the off-axis rim mismatch, ray-trace the full pupil grid
with batoid `traceFull` and record, for each ray, the **first surface that vignettes it**.
Counting losses among the rays that an ideal annular pupil `[R_in, R_out]` would pass
isolates the field-dependent vignetting batoid captures but danish's parametric mask only
approximates.

On-axis nothing in the annulus is lost; off-axis at R30_S21 a large fraction is clipped —
dominated by the **M1/M2/M3 mirror edges** (the beam footprint walks off the mirrors), with
a smaller **filter** contribution. This is exactly the field-dependent aperture that, if
imperfectly captured in danish's pupil mask, leaves the rim residuals in the fit.

In [ ]:
# ------------------------------------------------------------------
# Which optical surface vignettes the off-axis beam?  (batoid traceFull)
# ------------------------------------------------------------------
import collections


def vignetting_breakdown(cam_dz_mm, theta_deg, nx=500):
    """Fraction of nominal-annulus rays vignetted, and the per-surface breakdown, for a
    camera-only defocus at the given field. Returns (total_frac, {surface: frac})."""
    inst = getTaskInstrument("LSSTCam", detector_name)
    R_out, R_in = inst.radius, inst.radius * inst.obscuration
    tel = build_telescope(fiducial, cam_dz_mm, 0.0)
    rays = batoid.RayVector.asGrid(optic=tel, wavelength=wavelength,
                                   theta_x=np.deg2rad(theta_deg[0]),
                                   theta_y=np.deg2rad(theta_deg[1]), nx=nx)
    pr = np.hypot(rays.x, rays.y)            # entrance-pupil radius (m)
    first = np.array([""] * len(pr), dtype=object)
    vprev = np.zeros(len(pr), bool)
    for name, d in tel.traceFull(rays).items():
        vo = d["out"].vignetted
        newly = vo & ~vprev
        first[newly] = name
        vprev = vprev | vo
    ann = (pr >= R_in) & (pr <= R_out)       # rays an ideal annulus would pass
    lost = ann & vprev
    n_ann = int(ann.sum())
    frac = {s: int((lost & (first == s)).sum()) / n_ann
            for s in collections.Counter(first[lost])}
    return lost.sum() / n_ann, dict(sorted(frac.items(), key=lambda kv: -kv[1]))


R_edge = np.hypot(tx, ty)
cases = [("on-axis", (0.0, 0.0)), (f"R30_S21 ({R_edge:.2f}°)", (tx, ty))]
print("Giant camera-only +8 mm — vignetting of nominal-annulus rays:")
breakdowns = []
for lbl, th in cases:
    tot, frac = vignetting_breakdown(defocus_mm, th)
    breakdowns.append((lbl, tot, frac))
    print(f"\n  {lbl}: {100*tot:.1f}% of annulus rays vignetted")
    for s, f in frac.items():
        print(f"     {s:<16s} {100*f:5.1f}%")

lbl, tot, frac = breakdowns[-1]              # the off-axis case
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
names = list(frac.keys()); vals = [100 * frac[n] for n in names]
bars = ax.bar(names, vals, color="C3")
ax.set_ylabel("% of nominal-annulus rays vignetted")
ax.set_title(f"Off-axis vignetting by surface — giant camera +8 mm, {lbl}\n"
             f"total {100*tot:.1f}%  (on-axis: 0%)")
ax.tick_params(axis="x", rotation=30)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
plt.show()


<a id='maskedge'></a>
## Danish Mask vs Batoid Vignetting Boundary (pupil plane)

§9 showed batoid vignettes ~25% of the off-axis annulus at the mirror edges. danish's
pupil **does** model field-dependent edges — its `maskParams` are circles fit to the batoid
vignetting boundary (one per optical element), with a cubic-in-θ centre shift and radius
(ts_wep `maskUtils._fitEdges`/`fitMaskModel`). Here we compare the two **in danish's own
frame**: rays sampled on `optic.stopSurface` (the pupil radius then runs exactly
`R_inner→R_outer`), the batoid vignetting flag from a full trace, and danish's active mask
circles evaluated on the same points.

At R30 the circle mask reproduces batoid's boundary to **~98.6%**. The **inner**
(central-obscuration) edge is essentially exact at every azimuth. The residual is entirely
at the **outer** edge, where off-axis the true boundary becomes strongly **non-circular**
(it dips to ~3.36 m on the field-ward side vs 4.18 m opposite), while danish models each
edge as a single shifted **circle** of near-constant radius. danish's outer edge therefore
runs **too large** (mean ~+16 mm, up to ~+90 mm on the clipped side) — dominated by the
**filter** aperture, then M1. That outer over-extension is the pupil-plane origin of the
donut over-size in §8, i.e. the rim residual. Because it is a *shape* (non-circular)
mismatch, simply shrinking a circle's radius does not close it — an elliptical / higher-order
off-axis edge model for the filter and mirror apertures would be needed.

In [ ]:
# ------------------------------------------------------------------
# Danish mask vs batoid vignetting boundary, in ts_wep's pupil frame.
# Rays on optic.stopSurface (pupil radius R_inner..R_outer); danish circles are fit
# along +x then rotated via centre*thx/thr  (ts_wep maskUtils._fitEdges).
# ------------------------------------------------------------------
import collections

inst_mask = getTaskInstrument("LSSTCam", detector_name)   # mask is defocus-independent
mask_params = inst_mask.maskParams
r_pup = inst_mask.radius                                   # pupil outer radius (m)


def stop_pupil_sample(optic, thx_deg, thy_deg, nrad=200, naz=1200):
    """Sample the pupil as ts_wep fits the mask: rays on optic.stopSurface, plus the final
    vignetting flag and first vignetting surface from a full trace."""
    rays = batoid.RayVector.asPolar(optic=optic, wavelength=wavelength,
                                    theta_x=np.deg2rad(thx_deg), theta_y=np.deg2rad(thy_deg),
                                    nrad=nrad, naz=naz)
    pr = rays.copy().toCoordSys(optic.stopSurface.coordSys)
    optic.stopSurface.surface.intersect(pr)
    xP, yP = pr.x.copy(), pr.y.copy()
    first = np.array([""] * len(xP), dtype=object); vprev = np.zeros(len(xP), bool)
    for name, d in optic.traceFull(rays).items():
        vo = d["out"].vignetted; newly = vo & ~vprev
        first[newly] = name; vprev = vprev | vo
    return xP, yP, ~vprev, first


def danish_mask_circles(thx_deg, thy_deg):
    """Active danish mask circles at a field angle: (surface, edge, clear, cx, cy, r),
    replicating ts_wep maskUtils (polyval in θ_deg, centre along the field direction)."""
    thr = np.hypot(thx_deg, thy_deg)
    out = []
    for surf, edges in mask_params.items():
        if surf == "Spider_3D":
            continue
        for edge, p in edges.items():
            if thr < p["thetaMin"] or thr > p["thetaMax"]:
                continue
            r = np.polyval(p["radius"], thr); c = np.polyval(p["center"], thr)
            out.append((surf, edge, p["clear"], c * thx_deg / thr, c * thy_deg / thr, r))
    return out


def danish_pupil_mask(u, v, circles):
    keep = np.ones(len(u), bool)
    for surf, edge, clear, cx, cy, r in circles:
        inside = ((u - cx) ** 2 + (v - cy) ** 2) <= r ** 2
        keep &= inside if clear else ~inside
    return keep


xP, yP, bat_kept, bat_first = stop_pupil_sample(fiducial, tx, ty)
circ = danish_mask_circles(tx, ty)
dan_kept = danish_pupil_mask(xP, yP, circ)
agree = (dan_kept == bat_kept).mean()
permissive  = dan_kept & ~bat_kept            # danish keeps, batoid vignettes -> outer rim too wide
restrictive = ~dan_kept & bat_kept
perm_by_surf = collections.Counter(bat_first[permissive])
print(f"Danish mask vs batoid boundary at {detector_name} ({np.hypot(tx, ty):.2f}°): "
      f"agreement {100*agree:.1f}%")
print(f"  danish over-extends (keeps, batoid vignettes): {int(permissive.sum())} pts "
      f"-> {dict(perm_by_surf.most_common())}")
print(f"  danish over-blocks:                            {int(restrictive.sum())} pts")
print("  active danish edges (cx, cy, r) m:")
for s, e, clr, cx, cy, r in circ:
    print(f"    {s:<16s} {e:<6s} {'clear' if clr else 'obsc ':5s}  "
          f"c=({cx:+.3f},{cy:+.3f}) r={r:.4f}")


In [ ]:
# Pupil-plane overlay (left) and danish - batoid disagreement (right).
surf_colors = {"M1": "C0", "M2": "C1", "M3": "C2", "Filter_entrance": "C3", "L1_entrance": "C4"}
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13.5, 6.4), constrained_layout=True)
axL.scatter(xP[bat_kept], yP[bat_kept], s=1, color="0.8", label="batoid kept")
th = np.linspace(0, 2 * np.pi, 400)
for surf, edge, clear, cx, cy, r in circ:
    axL.plot(cx + r * np.cos(th), cy + r * np.sin(th),
             ls="-" if clear else "--", lw=1.6, color=surf_colors.get(surf, "k"),
             label=f"{surf} {edge} ({'clear' if clear else 'obsc'})")
axL.set_aspect("equal"); axL.set_xlim(-r_pup * 1.2, r_pup * 1.2); axL.set_ylim(-r_pup * 1.2, r_pup * 1.2)
axL.set_title(f"Pupil (stop frame): batoid kept + danish mask circles\nfield ({tx:.2f}, {ty:.2f})°",
              fontsize=10)
axL.set_xlabel("pupil x [m]"); axL.set_ylabel("pupil y [m]"); axL.legend(fontsize=7, loc="upper right")

axR.scatter(xP[bat_kept], yP[bat_kept], s=1, color="0.85")
axR.scatter(xP[permissive], yP[permissive], s=4, color="red",
            label=f"danish over-extends ({int(permissive.sum())})")
axR.scatter(xP[restrictive], yP[restrictive], s=4, color="blue",
            label=f"danish over-blocks ({int(restrictive.sum())})")
axR.set_aspect("equal"); axR.set_xlim(-r_pup * 1.2, r_pup * 1.2); axR.set_ylim(-r_pup * 1.2, r_pup * 1.2)
axR.set_title(f"danish − batoid disagreement (agreement {100*agree:.1f}%)", fontsize=10)
axR.set_xlabel("pupil x [m]"); axR.set_ylabel("pupil y [m]"); axR.legend(fontsize=8, loc="upper right")
plt.show()

# Pupil edge radius vs azimuth: batoid boundary (non-circular) vs danish circle mask.
rr = np.hypot(xP, yP); ph = np.degrees(np.arctan2(yP, xP))
nb = 72; be = np.linspace(-180, 180, nb + 1); pc = 0.5 * (be[:-1] + be[1:])
bo, do, bi, di = [], [], [], []
for k in range(nb):
    m = (ph >= be[k]) & (ph < be[k + 1])
    bk = m & bat_kept; dk = m & dan_kept
    bo.append(rr[bk].max() if bk.sum() > 3 else np.nan)
    do.append(rr[dk].max() if dk.sum() > 3 else np.nan)
    bi.append(rr[bk].min() if bk.sum() > 3 else np.nan)
    di.append(rr[dk].min() if dk.sum() > 3 else np.nan)
bo, do, bi, di = map(np.array, (bo, do, bi, di))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, 4.6), constrained_layout=True)
a1.plot(pc, bo, "o-",  ms=3, color="C0", label="batoid outer")
a1.plot(pc, do, "s--", ms=3, color="C0", mfc="none", label="danish outer")
a1.plot(pc, bi, "o-",  ms=3, color="C2", label="batoid inner")
a1.plot(pc, di, "s--", ms=3, color="C2", mfc="none", label="danish inner")
a1.set_xlabel("pupil azimuth [deg]"); a1.set_ylabel("edge radius [m]")
a1.set_title("Pupil edge vs azimuth (R30 field)"); a1.grid(alpha=0.3); a1.legend(fontsize=8)
a2.axhline(0, color="k", lw=0.6)
a2.plot(pc, (do - bo) * 1e3, "s-", color="C0", label="outer Δ")
a2.plot(pc, (di - bi) * 1e3, "s-", color="C2", label="inner Δ")
a2.set_xlabel("pupil azimuth [deg]"); a2.set_ylabel("danish − batoid edge [mm]")
a2.set_title(f"outer Δ: mean {1e3*np.nanmean(do - bo):+.0f}, "
             f"max {1e3*np.nanmax(do - bo):+.0f} mm; inner Δ ≈ 0")
a2.grid(alpha=0.3); a2.legend(fontsize=8)
plt.show()
